# 01c -- dim_fantasy_teams Seed

**Purpose**: Build `dim_fantasy_teams` from the league's Google Sheet owner registry.
The sheet is the single source of truth for team names, abbreviations, and
manager contacts.

**Key**: `team_key` (matches Google Sheet "Team ID" column: A01-A14, B01-B14)

**Source**: Google Sheet — must be set to "Anyone with the link can view"

**Divisions → Conferences**:
- Riddell → A (A01–A14)
- Wilson  → B (B01–B14)

**Cap columns**: this table only carries the two cap facts that are true
independent of the roster -- `original_cap` (`CFG.total_cap`, never changes)
and `reinvestment_cap` (in-season bonus cap charges, 0 at seed time).
Everything roster-derived (active roster salary, dead money, remaining cap)
used to be an ETL rollup cached here (02e); it's now computed live instead
-- from `Fact_FantasyTeams` via the `_Measures.tmdl` DAX measures in Power
BI, and from `fact_fantasy_teams.parquet` via `discord_bot/capmath.py` in
the bot. One formula, not a third cached copy.

**Outputs**:
- `data/dim_fantasy_teams.parquet`

In [1]:
# ---- Setup & Config (shared module) ----------------------------------------
# All config + path anchoring + helpers live in notebooks/etl_helpers.py.
# CFG.data_dir / DATA / REVIEW are anchored to the repo root -> writes always
# land in <repo>/data no matter the CWD this notebook runs from.
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
                         clean_player_name, generate_player_key, parse_height_to_inches)

import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import date
from dataclasses import dataclass, field


## 1 -- Load from Google Sheet

Re-run this notebook any time team names or manager emails change in the sheet.

In [2]:
# -- Pull team registry from Google Sheet ------------------------------------
# Sheet must be set to "Anyone with the link can view".
# pd.read_csv follows redirects automatically via urllib.
print(f"Loading team sheet: {CFG.team_sheet_csv_url}")
try:
    teams_raw = pd.read_csv(CFG.team_sheet_csv_url)
except Exception as e:
    raise RuntimeError(
        f"Failed to load team sheet. Confirm sheet is publicly viewable.\n{e}"
    ) from e

print(f"Rows loaded : {len(teams_raw)}")
print(f"Columns     : {list(teams_raw.columns)}")
teams_raw

Loading team sheet: https://docs.google.com/spreadsheets/d/1Fiz_KHH5bexSAHIfL0uVIqgHU6jTgnOmDs86kjR8TZc/export?format=csv&gid=178660131


Rows loaded : 28
Columns     : ['Division', 'Team Name', 'Team Abbreviation', 'Manager Email', 'Other Manager Email', 'Team ID', 'Fantrax-TeamId']


,Division,Team Name,Team Abbreviation,Manager Email,Other Manager Email,Team ID,Fantrax-TeamId
0,Riddell,"""National Anthem"" (R)",DegenMik,mvreeland6@gmail.com,NaN,A09,4uapm6ktmo32bach
1,Riddell,AC/DeeCeedeez Nuts (R),tempfool,juventude85@yahoo.com,NaN,A01,cw1na7n0moq8y8sq
2,Riddell,Big L (R),BIGL,benjaninja77@gmail.com,NaN,A10,pyjz79zdmn0kbl9b
3,Wilson,Black Magic Woman (W),BMW,justinacain24@gmail.com,NaN,B01,b3l7iznmmoboq9lg
4,Riddell,Brantley Gilbert (R),Thebusdr,j.milakovich@yahoo.com,NaN,A08,1vtj2q04mnax2yvd
5,Wilson,Bush (W),NT 7,jakeleer@icloud.com,NaN,B02,koqir027mo31l0cn
6,Riddell,Cyndi Lauper (R),NT 5,davismelanie1999@gmail.com,NaN,A02,ru6p9h06mo31l0c7
7,Wilson,Eminem (W),Shady,Valkyn KynVal,NaN,B06,s5bodsc1mn6pjqtc
8,Riddell,Flava Flav (R),Bjuenger,brenden.juengert@gmail.com,NaN,A03,oux5mrxgmn1ujei0
9,Wilson,Franz Ferdinand (W),FF,eduardo.ruizsanz@gmx.com,NaN,B05,cyq413lymnum51np


## 2 -- Build dim_fantasy_teams

In [3]:
# -- Column map: sheet -> dim_fantasy_teams -------------------------------------------
# Sheet columns: Division, Team ID, Team Name, Team Abbreviation,
#                Manager Email, Other Manager Email, Fantrax-TeamId
# Fantrax-TeamId is the league's authoritative teamId<->team_key identity
# (ADR-0005 locked column). Carrying it here lights up the ADR-0004 ledger join
# (teamId -> team_key in 02d) directly, retiring the name-match heuristic crosswalk.
_COL_MAP = {
    "Team ID":              "team_key",
    "Team Name":            "team_name",
    "Team Abbreviation":    "team_abbr",
    "Fantrax-TeamId":       "fantrax_team_id",
    "Division":             "division",
    "Manager Email":        "manager_email",
    "Other Manager Email":  "manager_email_2",
}

# Division -> conference letter (Riddell = A, Wilson = B)
_DIV_CONF = {"Riddell": "A", "Wilson": "B"}


def build_dim_fantasy_teams(raw: pd.DataFrame, cfg: LeagueConfig) -> pd.DataFrame:
    # Validate expected columns
    missing = [c for c in _COL_MAP if c not in raw.columns]
    if missing:
        raise ValueError(f"Sheet missing expected columns: {missing}")

    df = raw.rename(columns=_COL_MAP).copy()

    # Derive conference from division
    df["conference"] = df["division"].map(_DIV_CONF)
    unmapped_divs = df[df["conference"].isna()]["division"].unique()
    if len(unmapped_divs):
        raise ValueError(f"Unknown divisions -- add to _DIV_CONF: {list(unmapped_divs)}")

    # Normalise nullable email column
    df["manager_email_2"] = df["manager_email_2"].where(df["manager_email_2"].notna(), other=pd.NA)

    # Fantrax-TeamId must be present + unique (it's the ledger identity key).
    if df["fantrax_team_id"].isna().any():
        raise ValueError(
            f"Blank Fantrax-TeamId for: {list(df.loc[df['fantrax_team_id'].isna(), 'team_key'])}")
    if not df["fantrax_team_id"].is_unique:
        raise ValueError("Fantrax-TeamId values are not unique")

    # -- Cap state columns ---------------------------------------------------
    # original_cap/reinvestment_cap are the only true team-level cap facts --
    # everything roster-derived (active salary, dead money, remaining cap) is
    # computed live from fact_fantasy_teams instead of cached here (matches
    # the Power BI measures in _Measures.tmdl; see discord_bot/capmath.py for
    # the bot's live equivalent). 02e no longer rolls anything up into this
    # table.
    df["original_cap"]      = cfg.total_cap
    df["reinvestment_cap"]  = 0

    # -- Final column order -----------------------------------------------
    cols = [
        "team_key", "team_name", "team_abbr", "fantrax_team_id",
        "conference", "division",
        "manager_email", "manager_email_2",
        "original_cap", "reinvestment_cap",
    ]
    return df[cols].reset_index(drop=True)


dim_fantasy_teams = build_dim_fantasy_teams(teams_raw, CFG)

out_path = DATA / "dim_fantasy_teams.parquet"
dim_fantasy_teams.to_parquet(out_path, index=False)
print(f"dim_fantasy_teams: {len(dim_fantasy_teams)} teams -> {out_path}")
dim_fantasy_teams

dim_fantasy_teams: 28 teams -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\dim_fantasy_teams.parquet


,team_key,team_name,team_abbr,fantrax_team_id,conference,division,manager_email,manager_email_2,original_cap,reinvestment_cap
0,A09,"""National Anthem"" (R)",DegenMik,4uapm6ktmo32bach,A,Riddell,mvreeland6@gmail.com,NaN,300000000,0
1,A01,AC/DeeCeedeez Nuts (R),tempfool,cw1na7n0moq8y8sq,A,Riddell,juventude85@yahoo.com,NaN,300000000,0
2,A10,Big L (R),BIGL,pyjz79zdmn0kbl9b,A,Riddell,benjaninja77@gmail.com,NaN,300000000,0
3,B01,Black Magic Woman (W),BMW,b3l7iznmmoboq9lg,B,Wilson,justinacain24@gmail.com,NaN,300000000,0
4,A08,Brantley Gilbert (R),Thebusdr,1vtj2q04mnax2yvd,A,Riddell,j.milakovich@yahoo.com,NaN,300000000,0
5,B02,Bush (W),NT 7,koqir027mo31l0cn,B,Wilson,jakeleer@icloud.com,NaN,300000000,0
6,A02,Cyndi Lauper (R),NT 5,ru6p9h06mo31l0c7,A,Riddell,davismelanie1999@gmail.com,NaN,300000000,0
7,B06,Eminem (W),Shady,s5bodsc1mn6pjqtc,B,Wilson,Valkyn KynVal,NaN,300000000,0
8,A03,Flava Flav (R),Bjuenger,oux5mrxgmn1ujei0,A,Riddell,brenden.juengert@gmail.com,NaN,300000000,0
9,B05,Franz Ferdinand (W),FF,cyq413lymnum51np,B,Wilson,eduardo.ruizsanz@gmx.com,NaN,300000000,0


## 3 -- Validation

In [4]:
df = pd.read_parquet(DATA / "dim_fantasy_teams.parquet")
print(f"dim_fantasy_teams: {len(df)} rows, {len(df.columns)} columns")
print()

# Team count by conference / division
summary = df.groupby(["conference", "division"]).agg(
    teams=("team_key", "count"),
    missing_email=("manager_email", lambda x: x.isna().sum()),
).reset_index()
print("Teams by conference:")
print(summary.to_string(index=False))
print()

# Full roster display
display_cols = ["team_key", "team_name", "team_abbr", "conference", "division",
                "manager_email", "manager_email_2"]
print("Full team roster:")
print(df[display_cols].to_string(index=False))

dim_fantasy_teams: 28 rows, 10 columns

Teams by conference:
conference division  teams  missing_email
         A  Riddell     14              0
         B   Wilson     14              0

Full team roster:
team_key                 team_name team_abbr conference division              manager_email            manager_email_2
     A09     "National Anthem" (R)  DegenMik          A  Riddell       mvreeland6@gmail.com                        NaN
     A01    AC/DeeCeedeez Nuts (R)  tempfool          A  Riddell      juventude85@yahoo.com                        NaN
     A10                 Big L (R)      BIGL          A  Riddell     benjaninja77@gmail.com                        NaN
     B01     Black Magic Woman (W)       BMW          B   Wilson    justinacain24@gmail.com                        NaN
     A08      Brantley Gilbert (R)  Thebusdr          A  Riddell     j.milakovich@yahoo.com                        NaN
     B02                  Bush (W)      NT 7          B   Wilson        jakeleer